<a href="https://colab.research.google.com/github/jppeirce/DSC210-Foundations-of-Data-Science/blob/main/Notes/06-dictionaries_pandas/06-dictionaries_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 6: Dictionaries and pandas

**DSC 210 Foundations of Data Science**

References:
- [Hands-on Introduction to Data Science with Python](https://florian-huber.github.io/data_science_course/) (CC BY-NC-SA 4.0), and
- [pandas documentation](https://pandas.pydata.org/docs/)

```
ASK  ->  [ GET ]  ->  EXPLORE  ->  MODEL  ->  COMMUNICATE
```

So far we have used Python (Module 3), NumPy (Module 4), and seaborn (Module 5).

NumPy gave us fast arrays of numbers, but real datasets are **tables**: many rows,
several columns, and a mix of text and numbers in each row. 

This module gives us two tools that carry us through the **GET** stage and into **EXPLORE**: the Python
*dictionary* (a labeled lookup) and the pandas *DataFrame* (a spreadsheet in code).

## Learning objectives

1. Store labeled data in a **dictionary** and look values up by key.
2. Add, modify, delete, and iterate over key-value pairs.
3. Recognize a *record* as a dictionary and a *table* as a collection of records.
4. Build a pandas **DataFrame** from dictionaries and read one from a CSV file.
5. Inspect, select, filter, and sort a DataFrame.
6. Create new columns and compute descriptive statistics.

Our running dataset for this module is the **most-streamed songs on Spotify**. It is small
enough to read at a glance and familiar enough to reason about.

---
## 1. Dictionaries
---

### 1.1 The lookup problem

Suppose the campus radio station is tracking how many times some popular songs have been streamed (in billions). A first instinct, using what we know from Module 3, is two parallel lists:

`titles  = ['Blinding Lights', 'Shape of You', 'Riptide']`

`streams = [5.3, 4.8, 3.5]`

Simple question: *How many streams does 'Riptide' have?* 
With lists, we first have to find where the title lives, then read the matching position in the other list.

In [1]:
# RUN-TOGETHER

titles  = ['Blinding Lights', 'Shape of You', 'Riptide']
streams = [5.3, 4.8, 3.5]

# To look up Riptide we must find its position first, then index 
# the OTHER list.
idx = titles.index('Riptide')     # position of the title
print('position:', idx)
print('streams (billions):', streams[idx])

position: 2
streams (billions): 3.5


This works, but it is fragile:

- The two lists must stay in the **same order** forever. If we sort one and forget the other,
  every lookup silently returns the wrong song.
- Looking something up takes **two steps** (find the position, then index).

We want to ask for a value by its name, in one step. That is exactly what a **dictionary** does.

### 1.2 Dictionaries: lookup by key

**Definition.** A *dictionary* is an unordered collection of **key-value pairs**, written in
curly braces `{ }`. Each **key** points to a **value**, and we retrieve a value by writing the key in square brackets: `d[key]`.

- Keys must be unique and immutable (strings, ints, floats, booleans). Text keys are the most common.
- Values can be anything: numbers, strings, lists, or even other dictionaries.

Instead of two lists that must stay aligned, we bind each title to its streams figure directly.

In [6]:
# RUN-TOGETHER
# Curly braces, and 'key : value' pairs separated by commas.
song_streams = {'Blinding Lights': 5.3,
                'Shape of You': 4.8,
                'Riptide': 3.5}

# One-step lookup by name -- no positions, no second list.
print(song_streams['Riptide'])

# The whole dictionary:
print(song_streams)
print(type(song_streams))

3.5
{'Blinding Lights': 5.3, 'Shape of You': 4.8, 'Riptide': 3.5}
<class 'dict'>


### Activity 1

The station wants a quick "how far ahead is the leader?" stat.

A. Using two lookups and subtraction, print how many billion streams `'Blinding Lights'` is ahead of `'Riptide'`.

B. What does Python do if you ask for a title that is *not* a key, such as `song_streams['Levitating']`? 

(We fix this safely in the next section.)

In [ ]:
# Activity code:
# a. How far ahead is Blinding Lights over Riptide?
gap = song_streams[____] - song_streams[____]
print('Blinding Lights leads Riptide by', round(gap, 1), 'billion streams')

# b. PREDICT (comment only, do not run a missing-key lookup yet):
#    song_streams['Levitating'] would do what?
#    my guess: ____

### 1.3 Adding, modifying, and deleting pairs

A dictionary is *changeable*. We use the same square-bracket syntax to store a value:

- **Add** a new pair: `d[new_key] = value`
- **Modify** an existing value: `d[existing_key] = new_value` (same syntax, an existing key)
- **Check membership**: `key in d` returns `True`/`False`
- **Delete** a pair: `del d[key]`

Real data arrives incrementally and gets corrected. Adding, fixing, and removing
entries is the day-to-day reality of maintaining a dataset.

In [7]:
# FILL-IN

# ADD a song the station just started tracking.
song_streams['Die With a Smile'] = 3.5
print('after adding Die With a Smile:', song_streams)

# MEMBERSHIP test: is a key present?
print("'Die With a Smile' in song_streams? ->", 'Die With a Smile' in song_streams)
print("'Levitating' in song_streams?       ->", 'Levitating' in song_streams)

# MODIFY: streams keep climbing, so bump Die With a Smile up a little.
song_streams['Die With a Smile'] = 3.6
print('after updating Die With a Smile:', song_streams['Die With a Smile'])

# DELETE: the station stops tracking Shape of You.
del song_streams['Shape of You']
print('after deleting Shape of You:', song_streams)

after adding Die With a Smile: {'Blinding Lights': 5.3, 'Shape of You': 4.8, 'Riptide': 3.5, 'Die With a Smile': 3.5}
'Die With a Smile' in song_streams? -> True
'Levitating' in song_streams?       -> False
after updating Die With a Smile: 3.6
after deleting Shape of You: {'Blinding Lights': 5.3, 'Riptide': 3.5, 'Die With a Smile': 3.6}


### Activity 2

Real datasets get *corrected*, not just overwritten. Working from the current `song_streams`:

a. **Correction without hard-coding a number.** The station realizes `'Blinding Lights'` was over-counted by `0.4`. Lower it using its own current value (read the value, subtract, store it back).

b. **Add** `'Heat Waves'` at `3.7`.

c. **Safe lookup.** Complete an `if`/`else` that prints a song's streams *only if* the title is present, and otherwise prints "not tracked". Test it on `'Yellow'` (which is absent).

In [ ]:
# Activity 2
# a. read-modify-write: lower Blinding Lights by 0.4 using its CURRENT value (no typed number)
song_streams['Blinding Lights'] = song_streams[____] ____ 0.4

# b. add Heat Waves at 3.7
song_streams[____] = ____

# c. safe lookup guarded by `in`
title = 'Yellow'
if title ____ song_streams:
    print(title, '->', song_streams[title])
else:
    print(title, 'is not tracked')

print(song_streams)

In [4]:
# Examples - Adding data to dictionaries

# Add in a fictional country key-value pair
world['sealand']=0.000027

# We can check if 'sealand' is a key in our dictionary
print('sealand' in world)

# Oops, we need to update that value
world['sealand']=0.000028
print(world)

# Sealand is fake, remove it!
del(world['sealand'])
print(world)

True
{'afganistan': 30.55, 'albania': 2.77, 'algeria': 39.21, 'sealand': 2.8e-05}
{'afganistan': 30.55, 'albania': 2.77, 'algeria': 39.21}


### Lists vs Dictionaries

Overall, there are many similarities between Lists and Dictionaries
- Select, update, and remove with `[]`

Lists: sequence of values indexed by numbers

Dictionaries: sequence of values indexed by unique keys (any immutable type)

- If you have a collection of values where order matters, use a list
- If you need a lookup table (where looking up data is fast and easy), use a dictionary


## I heard you like dictionaries...
Dictionaries are built on key-value pairs
- keys must be unique and immutable
- values can be numeric, str, lists, or even other dictionaries
- We can chain hard brackets (`[]`) as needed for lookup

In [5]:
# Example - All the way down...

# Dictionary of dictionaries
europe = { 'spain': { 'capital':'madrid', 'population':46.77 },
           'france': { 'capital':'paris', 'population':66.03 },
           'germany': { 'capital':'berlin', 'population':80.62 },
           'norway': { 'capital':'oslo', 'population':5.084 } }


# Print out the capital of France
print(europe['france']['capital'])

# Create sub-dictionary data for italy
data = {'capital':'rome','population':59.83}

# Add data to europe under key 'italy'
europe["italy"] = data

# Print europe
print(europe)

paris
{'spain': {'capital': 'madrid', 'population': 46.77}, 'france': {'capital': 'paris', 'population': 66.03}, 'germany': {'capital': 'berlin', 'population': 80.62}, 'norway': {'capital': 'oslo', 'population': 5.084}, 'italy': {'capital': 'rome', 'population': 59.83}}


# Pandas

Pandas is a high-level data manipulation package
- great for tabular data (like spreadsheets)
- built on NumPy by Wes McKinney
- Store tabular data in a type called `DataFrame`
    - rows represent observations, have unique labels
    - columns represent features, have unique labels
- Typically installed with `import pandas as pd`



## DataFrames from Dictionaries

- Can be created from dictionaries
    - keys are column labels
    - values are data, column by column
    - BE CAREFUL HERE!! What should DataFrames look like?

- Typical function is `pandas.DataFrame(dict)`, but you may need to transpose the results

In [6]:
## DataFrames from Dictionaries

import pandas as pd
europe_df = pd.DataFrame(europe).T
print(europe_df)



        capital population
spain    madrid      46.77
france    paris      66.03
germany  berlin      80.62
norway     oslo      5.084
italy      rome      59.83


## DataFrames from External Sources
It is unlikely that you'll spend much time building dataframes from dictionaries.

Instead, you'll likely have external data files that you need to analyze.

We can read in common data types using Pandas. For example, we can use `pd.read_csv(path/to/awesome_data.csv)` to read .csv files.
- Always double-check that the data was read in correctly
- Common issue: row names (keys) are stored as a column (fix with an `index_col = 0` argument

# Example - Cars

Suppose we have vehicle data from different countries. Each observation corresponds to a country and the columns give information about the number of vehicles per capita, whether people drive left or right, and so on.



In [16]:
# Example - Cars

# Build up a dictionary
# First, some data (in lists)
names = ['United States', 'Australia', 'Japan', 'India', 'Russia', 'Morocco', 'Egypt']
dr =  [True, False, False, False, True, True, True]
cpc = [809, 731, 588, 18, 200, 70, 45]

# Create dictionary my_dict with three key:value pairs: my_dict
my_dict = {'country':names, 'drives_right':dr, 'cars_per_cap':cpc}

# Build a DataFrame cars from my_dict: cars
cars = pd.DataFrame(my_dict)

# Print cars
print(cars)

## Add row labels
# Definition of row_labels
row_labels = ['US', 'AUS', 'JPN', 'IN', 'RU', 'MOR', 'EG']

# Specify row labels of cars
cars.index = row_labels

print("")
print(cars)

         country  drives_right  cars_per_cap
0  United States          True           809
1      Australia         False           731
2          Japan         False           588
3          India         False            18
4         Russia          True           200
5        Morocco          True            70
6          Egypt          True            45

           country  drives_right  cars_per_cap
US   United States          True           809
AUS      Australia         False           731
JPN          Japan         False           588
IN           India         False            18
RU          Russia          True           200
MOR        Morocco          True            70
EG           Egypt          True            45


In [8]:
# Example - Cars
from google.colab import drive
drive.mount('/content/drive/')

# Starting from a .csv file
file_path = '/content/drive/MyDrive/Google Drive/Google Drive/Courses/DSC210/Notes/06-dictionaries_pandas/cars.csv'
cars = pd.read_csv(file_path, index_col=0)



# Print out cars
print(cars) # Note the row names and first column


Mounted at /content/drive/
           country  drives_right  cars_per_cap
US   United States          True           809
AUS      Australia         False           731
JPN          Japan         False           588
IN           India         False            18
RU          Russia          True           200
MOR        Morocco          True            70
EG           Egypt          True            45


# Indexing and Selecting Data

There are several ways to index or select data from pandas DataFrame objects
- Column indexing with square brackets (like lists and arrays)
    - Single brackets `[]` with a variable name yield a Series object
    - Double brackets `[[]]` with a variable name yield a DataFrame object
- Row indexing methods
    - Single brackets `[]` with an integer value (or range) yield a DataFrame object (a slice)
    - `.loc` for label-based indexing
    - `.iloc` for integer-based indexing
    - `[]` vs `[[]]` behavior for `.loc` and `.iloc` methods is similar to column indexing
    - Can be extended to select rows *and* columns


In [11]:
# Example - Cars
# Column Access
## Access by name
print("Accessing column by name (country) -- Series return:")
print(cars["country"]) # Note the print out and the last line
print(type(cars["country"])) # Pandas 'series' object, effectively a labeled 1D array

print("\n \n Accessing column by name (country) -- DataFrame return:")
print(cars[["country"]])
print(type(cars[["country"]])) # Pandas 'DataFrame' object



print("\n \n Selecting several columns:")
print(cars[["country", "drives_right"]])

# Row Access
## Access by index
# Print out first 3 observations
print("\n \n Row access by index:")
print(cars[0:3])

# Print out fourth, fifth and sixth observation
print(cars[3:6])



Accessing column by name (country) -- Series return:
US     United States
AUS        Australia
JPN            Japan
IN             India
RU            Russia
MOR          Morocco
EG             Egypt
Name: country, dtype: object
<class 'pandas.core.series.Series'>

 
 Accessing column by name (country) -- DataFrame return:
           country
US   United States
AUS      Australia
JPN          Japan
IN           India
RU          Russia
MOR        Morocco
EG           Egypt
<class 'pandas.core.frame.DataFrame'>

 
 Selecting several columns:
           country  drives_right
US   United States          True
AUS      Australia         False
JPN          Japan         False
IN           India         False
RU          Russia          True
MOR        Morocco          True
EG           Egypt          True

 
 Row access by index:
           country  drives_right  cars_per_cap
US   United States          True           809
AUS      Australia         False           731
JPN          Japan      

In [14]:
# Example - Cars

## Using loc and iloc
print("Row access by name -- Series return:")
print(cars.loc["JPN"])
print(type(cars.loc["JPN"]))

print("\n \n Row access by name -- DataFrame return:")
print(cars.loc[["JPN"]])
print(type(cars.loc[["JPN"]]))

print("\n \n Multiple rows by name:")
print(cars.loc[["JPN", "AUS"]])

print("\n \n Row and column access by name(s):")
print(cars.loc[["MOR"], ["drives_right"]])

print(cars.loc[["RU", "MOR"], ["country", "drives_right"]])


print("\n \n All row access (:) with column name, Series return")
print(cars.loc[:, "drives_right"])

# Print out drives_right column as DataFrame
print("\n \n All row access (:) with column name(s), DataFrame return")
print(cars.loc[:, ["drives_right"]])

# Print out cars_per_cap and drives_right as DataFrame
print(cars.loc[:, ["cars_per_cap", "drives_right"]])

Row access by name -- Series return:
country         Japan
drives_right    False
cars_per_cap      588
Name: JPN, dtype: object
<class 'pandas.core.series.Series'>

 
 Row access by name -- DataFrame return:
    country  drives_right  cars_per_cap
JPN   Japan         False           588
<class 'pandas.core.frame.DataFrame'>

 
 Multiple rows by name:
       country  drives_right  cars_per_cap
JPN      Japan         False           588
AUS  Australia         False           731

 
 Row and column access by name(s):
     drives_right
MOR          True
     country  drives_right
RU    Russia          True
MOR  Morocco          True

 
 All row access (:) with column name, Series return
US      True
AUS    False
JPN    False
IN     False
RU      True
MOR     True
EG      True
Name: drives_right, dtype: bool

 
 All row access (:) with column name(s), DataFrame return
     drives_right
US           True
AUS         False
JPN         False
IN          False
RU           True
MOR          Tru